In [1]:
import socket
import threading
from datetime import datetime
from queue import Queue
import tkinter as tk
from tkinter import ttk, scrolledtext, filedialog, messagebox
import csv
import ipaddress

class PortScanner:
    def __init__(self, root):
        self.root = root
        self.root.title("Advanced Port Scanner")
        self.root.geometry("900x700")
        self.root.resizable(True, True)
        
        self.scanning = False
        self.results = []
        
        self.create_widgets()
    
    def create_widgets(self):
        # Main frame
        main_frame = ttk.Frame(self.root, padding="10")
        main_frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # Target configuration
        target_frame = ttk.LabelFrame(main_frame, text="Target Configuration", padding="10")
        target_frame.grid(row=0, column=0, columnspan=2, sticky=(tk.W, tk.E), pady=5)
        
        ttk.Label(target_frame, text="Target(s):").grid(row=0, column=0, sticky=tk.W, padx=5)
        self.target_entry = ttk.Entry(target_frame, width=40)
        self.target_entry.grid(row=0, column=1, sticky=(tk.W, tk.E), padx=5)
        self.target_entry.insert(0, "127.0.0.1")
        
        ttk.Label(target_frame, text="Examples: 192.168.1.1 or 192.168.1.1-10 or 192.168.1.0/24", 
                 font=('Arial', 8)).grid(row=1, column=1, sticky=tk.W, padx=5)
        
        # Port configuration
        port_frame = ttk.LabelFrame(main_frame, text="Port Configuration", padding="10")
        port_frame.grid(row=1, column=0, columnspan=2, sticky=(tk.W, tk.E), pady=5)
        
        self.scan_mode = tk.StringVar(value="quick")
        ttk.Radiobutton(port_frame, text="Quick Scan (Common Ports)", 
                       variable=self.scan_mode, value="quick").grid(row=0, column=0, sticky=tk.W)
        ttk.Radiobutton(port_frame, text="Full Scan (1-1024)", 
                       variable=self.scan_mode, value="full").grid(row=1, column=0, sticky=tk.W)
        ttk.Radiobutton(port_frame, text="Custom Range", 
                       variable=self.scan_mode, value="custom").grid(row=2, column=0, sticky=tk.W)
        
        ttk.Label(port_frame, text="Start:").grid(row=2, column=1, padx=5)
        self.start_port = ttk.Entry(port_frame, width=10)
        self.start_port.grid(row=2, column=2, padx=5)
        self.start_port.insert(0, "1")
        
        ttk.Label(port_frame, text="End:").grid(row=2, column=3, padx=5)
        self.end_port = ttk.Entry(port_frame, width=10)
        self.end_port.grid(row=2, column=4, padx=5)
        self.end_port.insert(0, "100")
        
        # Options
        options_frame = ttk.LabelFrame(main_frame, text="Scan Options", padding="10")
        options_frame.grid(row=2, column=0, columnspan=2, sticky=(tk.W, tk.E), pady=5)
        
        self.grab_banner = tk.BooleanVar(value=False)
        ttk.Checkbutton(options_frame, text="Grab Service Banners", 
                       variable=self.grab_banner).grid(row=0, column=0, sticky=tk.W)
        
        ttk.Label(options_frame, text="Threads:").grid(row=0, column=1, padx=10)
        self.threads = ttk.Spinbox(options_frame, from_=10, to=500, width=10)
        self.threads.set(100)
        self.threads.grid(row=0, column=2)
        
        ttk.Label(options_frame, text="Timeout (s):").grid(row=0, column=3, padx=10)
        self.timeout = ttk.Spinbox(options_frame, from_=0.5, to=5, increment=0.5, width=10)
        self.timeout.set(1)
        self.timeout.grid(row=0, column=4)
        
        # Control buttons
        button_frame = ttk.Frame(main_frame)
        button_frame.grid(row=3, column=0, columnspan=2, pady=10)
        
        self.scan_btn = ttk.Button(button_frame, text="Start Scan", command=self.start_scan)
        self.scan_btn.grid(row=0, column=0, padx=5)
        
        self.stop_btn = ttk.Button(button_frame, text="Stop Scan", command=self.stop_scan, state=tk.DISABLED)
        self.stop_btn.grid(row=0, column=1, padx=5)
        
        ttk.Button(button_frame, text="Clear Results", command=self.clear_results).grid(row=0, column=2, padx=5)
        ttk.Button(button_frame, text="Export CSV", command=self.export_csv).grid(row=0, column=3, padx=5)
        ttk.Button(button_frame, text="Export TXT", command=self.export_txt).grid(row=0, column=4, padx=5)
        
        # Progress bar
        self.progress = ttk.Progressbar(main_frame, mode='determinate')
        self.progress.grid(row=4, column=0, columnspan=2, sticky=(tk.W, tk.E), pady=5)
        
        self.status_label = ttk.Label(main_frame, text="Ready to scan")
        self.status_label.grid(row=5, column=0, columnspan=2, sticky=tk.W)
        
        # Results area
        results_frame = ttk.LabelFrame(main_frame, text="Scan Results", padding="10")
        results_frame.grid(row=6, column=0, columnspan=2, sticky=(tk.W, tk.E, tk.N, tk.S), pady=5)
        
        self.results_text = scrolledtext.ScrolledText(results_frame, width=100, height=25, wrap=tk.WORD)
        self.results_text.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # Configure grid weights for resizing
        self.root.columnconfigure(0, weight=1)
        self.root.rowconfigure(0, weight=1)
        main_frame.columnconfigure(1, weight=1)
        main_frame.rowconfigure(6, weight=1)
        results_frame.columnconfigure(0, weight=1)
        results_frame.rowconfigure(0, weight=1)
    
    def parse_targets(self, target_str):
        """Parse target string and return list of IPs to scan."""
        targets = []
        target_str = target_str.strip()
        
        try:
            # Check for CIDR notation
            if '/' in target_str:
                network = ipaddress.ip_network(target_str, strict=False)
                targets = [str(ip) for ip in network.hosts()]
            # Check for range notation (e.g., 192.168.1.1-10)
            elif '-' in target_str:
                base, range_part = target_str.rsplit('.', 1)
                if '-' in range_part:
                    start, end = range_part.split('-')
                    for i in range(int(start), int(end) + 1):
                        targets.append(f"{base}.{i}")
                else:
                    targets.append(target_str)
            else:
                # Single IP or hostname
                targets.append(target_str)
        except Exception as e:
            messagebox.showerror("Error", f"Invalid target format: {e}")
            return []
        
        return targets
    
    def get_ports(self):
        """Get list of ports to scan based on mode."""
        mode = self.scan_mode.get()
        
        if mode == "quick":
            return [21, 22, 23, 25, 53, 80, 110, 143, 443, 445, 3306, 3389, 5432, 8080, 8443]
        elif mode == "full":
            return list(range(1, 1025))
        else:  # custom
            try:
                start = int(self.start_port.get())
                end = int(self.end_port.get())
                return list(range(start, end + 1))
            except ValueError:
                messagebox.showerror("Error", "Invalid port range")
                return []
    
    def scan_port(self, target, port, timeout):
        """Scan a single port."""
        try:
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            sock.settimeout(timeout)
            result = sock.connect_ex((target, port))
            sock.close()
            
            if result == 0:
                banner = None
                if self.grab_banner.get():
                    banner = self.get_banner(target, port, timeout)
                return (port, True, banner)
            return (port, False, None)
        except:
            return (port, False, None)
    
    def get_banner(self, target, port, timeout):
        """Try to grab service banner."""
        try:
            sock = socket.socket()
            sock.settimeout(timeout)
            sock.connect((target, port))
            banner = sock.recv(1024).decode().strip()
            sock.close()
            return banner[:100]
        except:
            return None
    
    def worker(self, target, timeout, queue, results):
        """Worker thread for scanning."""
        while not queue.empty() and self.scanning:
            port = queue.get()
            result = self.scan_port(target, port, timeout)
            results.append(result)
            queue.task_done()
    
    def scan_target(self, target, ports, num_threads, timeout):
        """Scan all ports on a target."""
        queue = Queue()
        results = []
        
        for port in ports:
            queue.put(port)
        
        threads = []
        for _ in range(min(num_threads, len(ports))):
            thread = threading.Thread(target=self.worker, args=(target, timeout, queue, results))
            thread.daemon = True
            thread.start()
            threads.append(thread)
        
        # Update progress
        total = len(ports)
        while not queue.empty() and self.scanning:
            completed = total - queue.qsize()
            progress = (completed / total) * 100
            self.progress['value'] = progress
            self.root.update_idletasks()
        
        queue.join()
        return sorted(results, key=lambda x: x[0])
    
    def get_service_name(self, port):
        """Get service name for port."""
        services = {
            21: "FTP", 22: "SSH", 23: "Telnet", 25: "SMTP", 53: "DNS",
            80: "HTTP", 110: "POP3", 143: "IMAP", 443: "HTTPS", 445: "SMB",
            3306: "MySQL", 3389: "RDP", 5432: "PostgreSQL", 8080: "HTTP-Proxy", 8443: "HTTPS-Alt"
        }
        return services.get(port, "Unknown")
    
    def start_scan(self):
        """Start the scanning process."""
        targets = self.parse_targets(self.target_entry.get())
        if not targets:
            return
        
        ports = self.get_ports()
        if not ports:
            return
        
        self.scanning = True
        self.scan_btn.config(state=tk.DISABLED)
        self.stop_btn.config(state=tk.NORMAL)
        self.results = []
        self.clear_results()
        
        # Run scan in separate thread
        thread = threading.Thread(target=self.run_scan, args=(targets, ports))
        thread.daemon = True
        thread.start()
    
    def run_scan(self, targets, ports):
        """Run the actual scan."""
        num_threads = int(self.threads.get())
        timeout = float(self.timeout.get())
        
        start_time = datetime.now()
        
        for target in targets:
            if not self.scanning:
                break
            
            self.status_label.config(text=f"Scanning {target}...")
            self.log(f"\n{'='*70}\nScanning: {target}\nTime: {datetime.now()}\n{'='*70}\n")
            
            try:
                results = self.scan_target(target, ports, num_threads, timeout)
                
                open_ports = [r for r in results if r[1]]
                
                if open_ports:
                    for port, is_open, banner in open_ports:
                        service = self.get_service_name(port)
                        msg = f"[+] {target}:{port} ({service}) - OPEN"
                        if banner:
                            msg += f"\n    Banner: {banner}"
                        self.log(msg)
                        self.results.append({
                            'target': target,
                            'port': port,
                            'service': service,
                            'status': 'OPEN',
                            'banner': banner or ''
                        })
                else:
                    self.log(f"No open ports found on {target}")
                
                self.log(f"\nSummary: {len(open_ports)} open ports found\n")
                
            except Exception as e:
                self.log(f"Error scanning {target}: {e}\n")
        
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        
        self.log(f"\n{'='*70}\nScan completed in {duration:.2f} seconds\n{'='*70}\n")
        self.status_label.config(text="Scan completed")
        self.progress['value'] = 100
        
        self.scanning = False
        self.scan_btn.config(state=tk.NORMAL)
        self.stop_btn.config(state=tk.DISABLED)
    
    def stop_scan(self):
        """Stop the scanning process."""
        self.scanning = False
        self.status_label.config(text="Scan stopped by user")
        self.scan_btn.config(state=tk.NORMAL)
        self.stop_btn.config(state=tk.DISABLED)
    
    def log(self, message):
        """Log message to results area."""
        self.results_text.insert(tk.END, message + "\n")
        self.results_text.see(tk.END)
        self.root.update_idletasks()
    
    def clear_results(self):
        """Clear results area."""
        self.results_text.delete(1.0, tk.END)
        self.progress['value'] = 0
    
    def export_csv(self):
        """Export results to CSV."""
        if not self.results:
            messagebox.showwarning("Warning", "No results to export")
            return
        
        filename = filedialog.asksaveasfilename(
            defaultextension=".csv",
            filetypes=[("CSV files", "*.csv"), ("All files", "*.*")]
        )
        
        if filename:
            try:
                with open(filename, 'w', newline='') as f:
                    writer = csv.DictWriter(f, fieldnames=['target', 'port', 'service', 'status', 'banner'])
                    writer.writeheader()
                    writer.writerows(self.results)
                messagebox.showinfo("Success", f"Results exported to {filename}")
            except Exception as e:
                messagebox.showerror("Error", f"Failed to export: {e}")
    
    def export_txt(self):
        """Export results to text file."""
        filename = filedialog.asksaveasfilename(
            defaultextension=".txt",
            filetypes=[("Text files", "*.txt"), ("All files", "*.*")]
        )
        
        if filename:
            try:
                with open(filename, 'w') as f:
                    f.write(self.results_text.get(1.0, tk.END))
                messagebox.showinfo("Success", f"Results exported to {filename}")
            except Exception as e:
                messagebox.showerror("Error", f"Failed to export: {e}")

def main():
    root = tk.Tk()
    app = PortScanner(root)
    root.mainloop()

if __name__ == "__main__":
    main()
